<a href="https://colab.research.google.com/github/marikhakhishvili/Facial-Expression-Recognition-Challenge/blob/main/model3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle wandb onnx -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 8.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
! mkdir ~/.kaggle

In [4]:
!cp /content/drive/MyDrive/cs231n/assignments/assignment4/kaggle.json ~/.kaggle/kaggle.json

In [5]:
! chmod 600 ~/.kaggle/kaggle.json

download competition data

In [6]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

100% 285M/285M [00:02<00:00, 147MB/s]



In [7]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
  inflating: example_submission.csv  
  inflating: fer2013.tar.gz          
  inflating: icml_face_data.csv      
  inflating: test.csv                
  inflating: train.csv               


In [8]:
import wandb
!wandb login --relogin

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Preprocessing

In [11]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

df = pd.read_csv('./train.csv')


def pixels_to_image_array(pixels_str):
    """
    Converts a space-separated pixel string into a normalized
    1 x 48 x 48 NumPy array suitable for PyTorch.
    """

    # Convert string to NumPy array of floats
    pixels = np.array(pixels_str.split(), dtype=np.float32)

    # Reshape into a 48x48 image
    image = pixels.reshape(48, 48)

    # Normalize pixel values from [0, 255] to [0, 1]
    image = image / 255.0

    # Add channel dimension: (48, 48) -> (1, 48, 48)
    image = np.expand_dims(image, axis=0)

    return image

# Apply preprocessing to every image
df["pixels"] = df["pixels"].apply(pixels_to_image_array)

# Check the shape of the first image
print(df["pixels"].iloc[0].shape)
# Expected output: (1, 48, 48)

(1, 48, 48)


In [12]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

X = np.stack(df["pixels"].values)   # shape: (N, 1, 48, 48)
y = df["emotion"].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
print("\nEmotion distribution in New Training Set:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nEmotion distribution in Validation Set:")
print(pd.Series(y_val).value_counts(normalize=True))


Emotion distribution in New Training Set:
3    0.251317
6    0.172944
4    0.168241
2    0.142683
0    0.139156
5    0.110463
1    0.015196
Name: proportion, dtype: float64

Emotion distribution in Validation Set:
3    0.251306
6    0.172936
4    0.168234
2    0.142807
0    0.139150
5    0.110414
1    0.015152
Name: proportion, dtype: float64


In [14]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

Model

In [16]:
import torch.nn as nn

class CNN_Model3(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [17]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return total_loss / len(loader), correct / total

Training

In [18]:
wandb.init(
    project="fer2013-assignment",
    name="model3_dropout_lr5e-4_bs64",
    config={
        "model": "CNN_Model3",
        "lr": 5e-4,
        "batch_size": 64,
        "epochs": 10,
        "weight_decay": 1e-4,
        "optimizer": "Adam",
        "dropout1": 0.5,
        "dropout2": 0.3
    }
)

config = wandb.config

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mkhak23 (mkhak23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model3().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.lr,
    weight_decay=config.weight_decay
)

In [20]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

Epoch 1/10
Train Loss: 1.7090, Train Acc: 0.3079
Val Loss: 1.5680, Val Acc: 0.4124
Epoch 2/10
Train Loss: 1.5135, Train Acc: 0.4071
Val Loss: 1.4251, Val Acc: 0.4406
Epoch 3/10
Train Loss: 1.4240, Train Acc: 0.4460
Val Loss: 1.3461, Val Acc: 0.4944
Epoch 4/10
Train Loss: 1.3574, Train Acc: 0.4770
Val Loss: 1.2878, Val Acc: 0.5059
Epoch 5/10
Train Loss: 1.3023, Train Acc: 0.4995
Val Loss: 1.2632, Val Acc: 0.5106
Epoch 6/10
Train Loss: 1.2586, Train Acc: 0.5147
Val Loss: 1.2459, Val Acc: 0.5268
Epoch 7/10
Train Loss: 1.2210, Train Acc: 0.5343
Val Loss: 1.2200, Val Acc: 0.5496
Epoch 8/10
Train Loss: 1.1896, Train Acc: 0.5430
Val Loss: 1.2028, Val Acc: 0.5623
Epoch 9/10
Train Loss: 1.1576, Train Acc: 0.5584
Val Loss: 1.1931, Val Acc: 0.5594
Epoch 10/10
Train Loss: 1.1173, Train Acc: 0.5728
Val Loss: 1.1722, Val Acc: 0.5601


In [21]:
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▄▅▅▆▆▇▇██
train_loss,█▆▅▄▃▃▂▂▁▁
val_acc,▁▂▅▅▆▆▇███
val_loss,█▅▄▃▃▂▂▂▁▁
epoch,10
train_acc,0.57278
train_loss,1.11727
val_acc,0.56008
val_loss,1.17222


Training with different parameters